In [ ]:
import argparse
import time
import numpy as np
import wandb

from reasoning.tools.utils import load_model_with_vllm
from datasets import load_dataset
from vllm import SamplingParams
from reasoning.inference.majority import MajorityInference
from reasoning.tools.logger import load_config, apply_config

from reasoning.evaluator.math_grader import math_equal, extract_answer

from reasoning.models.model import ValueModel_qwen


config = load_config("/ssdscratch/byuan48/efficient_reasoning/configs/experiments_418/development.yaml")
apply_config(config) 
print(config)

# load dataset and model
dataset = load_dataset(config["data_path"])
dataset = dataset['test']
model_name = config["model_name"]

policy_model, tokenizer = load_model_with_vllm(model_name, task='auto', tensor_parallel_size=len(config["cuda_device_ids"]), gpu_memory_utilization=0.8)
tokenizer.pad_token = tokenizer.eos_token 
reward_model = ValueModel_qwen(device = "auto") # default one
start_problem_index = config.get("begin_problem_index", 0)
end_problem_index = config.get("end_problem_index", len(dataset)-1)
dataset = dataset.select(range(start_problem_index, end_problem_index)) # need to check this

# inference hyperparameters
num_return_sequences = config["num_return_sequences"]
assert num_return_sequences == 1
max_new_tokens = config["max_new_tokens"]
temperature = config["temperature"]
top_p = config["top_p"]
system_prompt = config["system_prompt"]

# SamplingTree parameters
sampling_method = config["sampling_method"]
samplingTree_temperature = config["samplingTree_temperature"]
beam_width = config["beam_width"]
max_steps = config["max_steps"]
threshold = config["threshold"]
ORM_type = config["ORM_type"]
# name of the results file
config_name = config["config_name"]

# Also set the seed for the sampling params 
sampling_params = SamplingParams(
    temperature=temperature,
    max_tokens=max_new_tokens,
    n=num_return_sequences,
    top_p=top_p,
    stop_token_ids=[tokenizer.eos_token_id],
    skip_special_tokens = True,
    include_stop_str_in_output = False,
    seed  = config["seed"]
) # shouldn't set seed for random sampling


In [2]:
import importlib
import reasoning.inference.tree
tree_module = importlib.reload(reasoning.inference.tree)
Tree = tree_module.Tree

In [ ]:

start_time = time.time()
right_count = 0
num_generated_tokens = 0
error_steps_count_all = 0
print(f'length of the dataset: {len(dataset)}')
for i in range(0, len(dataset)):
    question = dataset['problem'][i]
    answer = dataset['solution'][i]        

    tree = Tree(system_prompt, question, policy_model, reward_model, sampling_method, config_name, sampling_params, samplingTree_temperature, beam_width, threshold, ORM_type)
    for _ in range(max_steps):
        path = tree.generate_next_trajectory() 
        tree.paths.append(path)
    majority_inference = MajorityInference(policy_model=policy_model, tokenizer=tokenizer, sampling_params=sampling_params, config_name=config_name, reward_model=reward_model, method = 'weighted_majority', ORM_type=ORM_type)
    all_solutions  = [path.solutions for path in tree.explored_paths]
    rewards = [tree.get_reward(path.scores) for path in tree.explored_paths]
    
    solution = majority_inference.weighted_majority_vote(all_solutions, rewards)   
    extracted_answer = extract_answer(answer)
    is_correct = math_equal(extracted_answer, solution)
    if extracted_answer is None:
        raise ValueError('extracted answer is None')
    if is_correct:
        right_count += 1
    num_generated_tokens += tree.num_generated_tokens
    print("--------------------------------")
    print(f'question: {question}')
    print(f'solution: {solution}')
    print(f'extracted answer: {extracted_answer}')
    print(f'is correct: {is_correct}')
    print(f'Accuracy: {right_count/(i+1)}')
    wandb.log({"Accuracy": right_count/(i+1)}, step=i)
    print(f'Number of generated tokens: {tree.num_generated_tokens}')
    # print out the error steps count
    error_steps_count = 0
    for path in tree.explored_paths:
        if len(path.steps) <= 1:
            error_steps_count += 1
    error_steps_count_all += error_steps_count
    print(f'Error steps count: {error_steps_count_all}')
    wandb.log({"Error steps count": error_steps_count_all}, step=i)
    print("--------------------------------")


wandb.log({"Accuracy": right_count/len(dataset)})
wandb.log({"Number of generated tokens": num_generated_tokens/len(dataset)})
end_time = time.time()
print("Time taken: {} seconds".format(end_time - start_time))

wandb.finish()

Processed prompts: 100%|██████████| 1/1 [00:05<00:00,  5.52s/it, est. speed input: 62.85 toks/s, output: 245.07 toks/s]


first_score_index: 0
scores: [0.00421143 0.30273438]
path.steps: ['## Step 1: To rotate a complex number $z$ around another complex number $c$ by an angle $\\theta$, we can use the formula $w = z \\cdot e^{i\\theta}$.\n## Step 2: First, we need to express $z$ and $c$ in polar form. The polar form of a complex number $z = a + bi$ is given by $re^{i\\theta}$, where $r = \\sqrt{a^2 + b^2}$ and $\\theta = \\tan^{-1}\\left(\\frac{b}{a}\\right)$.\n## Step 3: The polar form of $z = 2 + \\sqrt{2} - (3 + 3 \\sqrt{2})i$ is $z = 2\\sqrt{2}e^{i\\tan^{-1}\\left(\\frac{\\sqrt{2}}{3}\\right)}$.\n## Step 4: The polar form of $c = 2 - 3i$ is $c = \\sqrt{2^2 + (-3)^2}e^{i\\tan^{-1}\\left(\\frac{-3}{2}\\right)} = 2\\sqrt{13}e^{i\\tan^{-1}\\left(\\frac{-3}{2}\\right)}$.\n## Step 5: To rotate $z$ around $c$ by $\\frac{\\pi}{4}$ counter-clockwise, we use the formula $w = z \\cdot e^{i\\frac{\\pi}{4}}$.\n## Step 6: We can rewrite $e^{i\\frac{\\pi}{4}}$ as $\\cos\\left(\\frac{\\pi}{4}\\right) + i\\sin\\left(\

Processed prompts: 100%|██████████| 4/4 [00:03<00:00,  1.15it/s, est. speed input: 397.65 toks/s, output: 692.74 toks/s]


found solutions: ['-1 + i(2 - \\sqrt{2})', '-1 + i(2 - \\sqrt{2})', '\\frac{-1 - i\\sqrt{2}}{2}', '\\frac{-1 - i\\sqrt{2}}{2}']
rewards: [0.00012370274999302336, 0.00012370274999302336, 3.900367599586038e-05, 3.900367599586038e-05]
--------------------------------
question: Let $z = 2 + \sqrt{2} - (3 + 3 \sqrt{2})i$, and let $c = 2 - 3i$.  Let $w$ be the result when $z$ is rotated around $c$ by $\frac{\pi}{4}$ counter-clockwise.

[asy]
unitsize(0.6 cm);

pair C, W, Z;

Z = (2 + sqrt(2), -3 - 3*sqrt(2));
C = (2,-3);
W = rotate(45,C)*(Z);

draw(Z--C--W);

dot("$c$", C, N);
dot("$w$", W, SE);
dot("$z$", Z, S);
label("$\frac{\pi}{4}$", C + (0.6,-1));
[/asy]

Find $w.$
solution: -1 + i(2 - \sqrt{2})
extracted answer: 6 - 5i
is correct: False
Accuracy: 0.375
Number of generated tokens: 3771
Error steps count: 0
--------------------------------


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s, est. speed input: 229.76 toks/s, output: 239.43 toks/s]


...

found solutions: ['0', '0', '0', '0']
rewards: [0.00729006796559295, 0.0053940193608215115, 0.0053940193608215115, 0.004508160586461085]
--------------------------------
question: If $n \equiv 2 \pmod{7}$, then find the remainder when $(n + 2)(n + 4)(n + 6)$ is divided by 7.
solution: 0
extracted answer: 3
is correct: False
Accuracy: 0.3611111111111111
Number of generated tokens: 1152
Error steps count: 0
--------------------------------


Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.21s/it, est. speed input: 47.10 toks/s, output: 245.02 toks/s]


first_score_index: 1
scores: [0.9921875  0.18652344 0.01507568 0.9765625  0.9453125  0.8515625
 0.7734375  0.97265625 0.10107422 0.00680542 0.51953125 0.53515625
 0.87109375 0.92578125 0.9296875  0.43164062 0.8828125 ]
path.steps: ['## Step 1: Expand the equation\nWe start by expanding both sides of the given equation to simplify it.', '## Step 2: Factor the equation\nThe expanded form of the equation is:\n\\[(x - 3)^3 + (x - 7)^3 = (2x - 10)^3\\]\n\\[(x^3 - 9x^2 + 27x - 27) + (x^3 - 21x^2 + 147x - 343) = (8x^3 - 60x^2 + 200x - 1000)\\]', '## Step 3: Combine like terms\nCombine the like terms on the left side of the equation:\n\\[(2x^3 - 9x^2 + 27x - 27) + (x^3 - 21x^2 + 147x - 343) = (8x^3 - 60x^2 + 200x - 1000)\\]\n\\[3x^3 - 30x^2 + 174x - 320 = 8x^3 - 60x^2 + 200x - 1000\\]', '## Step 4: Move all terms to one side of the equation\nSubtract $8x^3 - 60x^2 + 200x - 1000$ from both sides of the equation to move all terms to the left side:\n\\[3x^3 - 30x^2 + 174x - 320 - (8x^3 - 60x^2 + 

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

: 